# EXP-RANPAC-OFFICIAL-REPLICATION-WITH-SOKAC: REPLICATE 100% REPO GỐC RANPAC (NEURIPS 2023) & ĐỐI CHỨNG SO-KAC
### MỤC TIÊU THỰC NGHIỆM: ĐỐI CHỨNG CÔNG BẰNG TUYỆT ĐỐI THEO CHUẨN PHƯƠNG ÁN A (GIT CLONE OFFICIAL REPO)
* **Ngày cập nhật:** 2026-10-29
* **Chiến lược kiến trúc (Phương án A):**
  1. **100% Mã Nguồn Gốc Tác Giả:** Trực tiếp clone và chạy repository chính thức `McDonnell-Research-Lab/RanPAC.git`. Không tự viết lại bất kỳ hàm trích xuất, huấn luyện, giải ma trận Ridge (`optimise_ridge_parameter`) hay hàm đánh giá (`eval_task`).
  2. **Bộ Điều Phối Đường Dẫn Dataset Linh Hoạt:** Tự động tìm kiếm tập dữ liệu trên Kaggle (`/kaggle/input/...`) hoặc Local máy trạm mà không cần copy hay di chuyển file.
  3. **Module Mở Rộng SO-KAC Ridge 6-Head (Main Proposal - 6x2,304d):** Tích hợp trực tiếp kiến trúc đề xuất chính thức của bài báo: 6 Heads x 2,304 dims (12 khối trực giao Haar QR ANOVA + hàm phi tuyến Sobolev 3-ReLU knots + skip connection). Giải song song 6 ma trận Gram nhỏ 2304x2304 với RAM ma trận Gram chỉ 121.5 MB (-68.2%) và độ phức tạp Cholesky giảm 92.7%. Giữ nguyên 100% pipeline trích xuất [CLS], tính toán ma trận Gram, tìm $\lambda$ tối ưu và đánh giá kết quả của tác giả.
  4. **Đối Chứng Độc Lập 2 Bộ Weights Pretrained ViT-B/16:**
     - `IN21K`: `vit_base_patch16_224_in21k` (Huấn luyện trước trên ImageNet-21K)
     - `IN1K`: `vit_base_patch16_224` (Fine-tuned trên ImageNet-1K)
  5. **Báo cáo chuẩn mực:** Đo và in đầy đủ 2 chỉ số học thuật cốt lõi **Last-Task Accuracy ($A_T$)** và **Average Incremental Accuracy ($A_{\text{avg}}$)** qua 3 Seeds ngẫu nhiên `[0, 1, 2]`.

In [ ]:
# ==============================================================================
# CELL 2: CÀI ĐẶT MÔI TRƯỜNG & CLONE REPO CHÍNH THỨC CỦA TÁC GIẢ RANPAC
# ==============================================================================
import os, sys, shutil

IS_KAGGLE = os.path.exists("/kaggle")
print(f"[*] Môi trường thực thi: {'Kaggle GPU' if IS_KAGGLE else 'Local/Server'}")

# Cài đặt timm nếu chưa có
try:
    import timm
except ImportError:
    !pip install -q timm
    import timm

# Clone repo chính thức của tác giả RanPAC
REPO_DIR = "RanPAC"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repo chính thức từ McDonnell-Research-Lab/RanPAC...")
    !git clone https://github.com/McDonnell-Research-Lab/RanPAC.git
    print("[OK] Clone repo RanPAC thành công!")
else:
    print(f"[OK] Đã tìm thấy thư mục repo tại: {REPO_DIR}")

In [ ]:
# ==============================================================================
# CELL 3: BỘ TỰ ĐỘNG ĐỊNH VỊ ĐƯỜNG DẪN DATASET (KAGGLE & LOCAL)
# ==============================================================================
# Khôi phục utils/data.py về trạng thái sạch gốc trước khi patch
os.system(f"cd {REPO_DIR} && git checkout -- utils/data.py 2>/dev/null")

# Cập nhật utils/data.py để tự động trỏ đúng đường dẫn dataset trên Kaggle
data_py_path = os.path.join(REPO_DIR, "utils", "data.py")

with open(data_py_path, "r", encoding="utf-8") as f:
    orig_code = f.read()

# Mã nguồn chắp vá thông minh cho utils/data.py
patch_code = '''
# --- DLP-302M MULTI-CANDIDATE DATASET PATH RESOLVER ---
import os
import numpy as np
from torchvision import datasets, transforms
from utils.toolkit import split_images_labels

def _resolve_dir(candidates, default_path):
    for c in candidates:
        if os.path.exists(c):
            return c
    return default_path

# Override cars download_data
def _cars_download_data(self):
    root = _resolve_dir([
        "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
        "/kaggle/input/stanford-car-dataset-by-classes-folder/car_data/car_data",
        "d:/Documents/Project DLP 302m/data/cars",
        "./data/cars"
    ], "./data/cars")
    tr_dir = os.path.join(root, "train") if os.path.exists(os.path.join(root, "train")) else root
    te_dir = os.path.join(root, "test") if os.path.exists(os.path.join(root, "test")) else root
    tr_dset = datasets.ImageFolder(tr_dir)
    te_dset = datasets.ImageFolder(te_dir)
    self.train_data, self.train_targets = split_images_labels(tr_dset.imgs)
    self.test_data, self.test_targets = split_images_labels(te_dset.imgs)
cars.download_data = _cars_download_data

# Override CUB download_data
def _cub_download_data(self):
    root = _resolve_dir([
        "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
        "/kaggle/input/cub-200-2011-by-classes-folder",
        "/kaggle/input/cub200/CUB_200_2011",
        "d:/Documents/Project DLP 302m/data/cub200",
        "./data/cub"
    ], "./data/cub")
    tr_dir = os.path.join(root, "train")
    te_dir = os.path.join(root, "test")
    if os.path.exists(tr_dir) and os.path.exists(te_dir):
        tr_dset = datasets.ImageFolder(tr_dir)
        te_dset = datasets.ImageFolder(te_dir)
        self.train_data, self.train_targets = split_images_labels(tr_dset.imgs)
        self.test_data, self.test_targets = split_images_labels(te_dset.imgs)
    else:
        img_dir = os.path.join(root, "images") if os.path.exists(os.path.join(root, "images")) else root
        full = datasets.ImageFolder(img_dir)
        rng = np.random.RandomState(42)
        idx = rng.permutation(len(full))
        n_tr = int(0.5 * len(full))
        self.train_data = np.array([full.imgs[i][0] for i in idx[:n_tr]])
        self.train_targets = np.array([full.imgs[i][1] for i in idx[:n_tr]])
        self.test_data = np.array([full.imgs[i][0] for i in idx[n_tr:]])
        self.test_targets = np.array([full.imgs[i][1] for i in idx[n_tr:]])
CUB.download_data = _cub_download_data

# Override iCIFAR224 download_data
def _cifar_download_data(self):
    cifar_dir = _resolve_dir([
        "/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
        "/kaggle/input/datasets/thohintrnhngc/cifar-100-python",
        "./data"
    ], "./data")
    if os.path.exists(os.path.join(cifar_dir, "train")):
        cifar_dir = os.path.dirname(cifar_dir)
    try:
        tr_ds = datasets.cifar.CIFAR100(cifar_dir, train=True, download=False)
        te_ds = datasets.cifar.CIFAR100(cifar_dir, train=False, download=False)
    except Exception:
        tr_ds = datasets.cifar.CIFAR100("./data", train=True, download=True)
        te_ds = datasets.cifar.CIFAR100("./data", train=False, download=True)
    self.train_data, self.train_targets = tr_ds.data, np.array(tr_ds.targets)
    self.test_data, self.test_targets = te_ds.data, np.array(te_ds.targets)
iCIFAR224.download_data = _cifar_download_data

# Override iImageNetR download_data
def _imagenetr_download_data(self):
    root = _resolve_dir([
        "/kaggle/input/datasets/thohintrnhngc/imagenet-r",
        "/kaggle/input/imagenet-r/imagenet-r",
        "d:/Documents/Project DLP 302m/data/imagenet-r",
        "./data/imagenet-r"
    ], "./data/imagenet-r")
    tr_dir = os.path.join(root, "train")
    te_dir = os.path.join(root, "test")
    if os.path.exists(tr_dir) and os.path.exists(te_dir):
        tr_dset = datasets.ImageFolder(tr_dir)
        te_dset = datasets.ImageFolder(te_dir)
        self.train_data, self.train_targets = split_images_labels(tr_dset.imgs)
        self.test_data, self.test_targets = split_images_labels(te_dset.imgs)
    else:
        full = datasets.ImageFolder(root)
        rng = np.random.RandomState(42)
        idx = rng.permutation(len(full))
        n_tr = int(0.8 * len(full))
        self.train_data = np.array([full.imgs[i][0] for i in idx[:n_tr]])
        self.train_targets = np.array([full.imgs[i][1] for i in idx[:n_tr]])
        self.test_data = np.array([full.imgs[i][0] for i in idx[n_tr:]])
        self.test_targets = np.array([full.imgs[i][1] for i in idx[n_tr:]])
iImageNetR.download_data = _imagenetr_download_data
'''

if "DLP-302M MULTI-CANDIDATE" not in orig_code:
    with open(data_py_path, "a", encoding="utf-8") as f:
        f.write("\n" + patch_code + "\n")
    print("[OK] Đã gắn bộ điều phối dataset Kaggle/Local vào RanPAC/utils/data.py thành công!")
else:
    print("[OK] utils/data.py đã được cấu hình sẵn sàng!")

In [ ]:
# ==============================================================================
# CELL 4: TÍCH HỢP MODULE SO-KAC & THAM SỐ DÒNG LỆNH VÀO RanPAC
# ==============================================================================
# Khôi phục các file về trạng thái sạch gốc trước khi patch
os.system(f"cd {REPO_DIR} && git checkout -- main.py inc_net.py RanPAC.py trainer.py 2>/dev/null")

# 1. Patch main.py: Thêm các tham số --sokac, --backbone, --seeds
main_py_path = os.path.join(REPO_DIR, "main.py")
new_main_code = '''import pandas as pd
import argparse
from trainer import train

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('-i', type=int, default=4, help='Experiment ID in csv')
    parser.add_argument('-d', type=str, default='cars', help='Dataset name')
    parser.add_argument('--backbone', type=str, default=None, help='Backbone model name')
    parser.add_argument('--sokac', action='store_true', help='Use SO-KAC 6-Head Ridge (6x2304d)')
    parser.add_argument('--seeds', type=int, nargs='+', default=None, help='Seeds list (e.g. 0 1 2)')
    a = parser.parse_args()

    exps = pd.read_csv('./args/' + a.d + '_publish.csv')
    args = exps[exps['ID'] == a.i].to_dict('records')[0]
    
    # Ghi đè tham số tùy biến nếu người dùng truyền vào
    if a.backbone:
        args['convnet_type'] = a.backbone
    if a.sokac:
        args['use_sokac'] = True
    if a.seeds:
        args['seed'] = a.seeds
    else:
        args['seed'] = [args['seed']]
        
    args['device'] = [args['device']]
    args['do_not_save'] = False
    
    # Định danh ID duy nhất để không bị ghi đè file kết quả
    suffix = ('_sokac_6head' if a.sokac else '_ranpac_10k') + ('_' + args['convnet_type'])
    args['run_tag'] = str(args['ID']) + suffix
    print(f"[*] Bắt đầu thực nghiệm: Dataset={a.d} | RunTag={args['run_tag']} | Seeds={args['seed']}")
    
    train(args)

if __name__ == '__main__':
    main()
'''
with open(main_py_path, "w", encoding="utf-8") as f:
    f.write(new_main_code)
print("[OK] Đã cập nhật RanPAC/main.py với tham số --sokac, --backbone, --seeds!")

# 2. Patch inc_net.py: Thêm SOKAC_6Head_Projector và mở rộng CosineLinear
inc_net_path = os.path.join(REPO_DIR, "inc_net.py")
with open(inc_net_path, "r", encoding="utf-8") as f:
    inc_net_code = f.read()

sokac_class_code = '''
# --- DLP-302M SO-KAC 6-HEAD RIDGE PROJECTOR MODULE (MAIN PROPOSAL) ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SOKAC_6Head_Projector(nn.Module):
    def __init__(self, in_features=768, num_heads=6, orth_dim=1536, seed=1993):
        super().__init__()
        self.in_features = in_features
        self.num_heads = num_heads
        self.orth_dim = orth_dim
        self.head_dim = in_features + orth_dim # 2304
        self.kac_gain = 1.35
        self.alpha_spline = 0.45
        self.register_buffer("global_rms", torch.tensor(0.0, dtype=torch.float32))

        g = torch.Generator().manual_seed(seed)
        total_orth = num_heads * orth_dim # 9216
        num_qr = (total_orth + in_features - 1) // in_features # 12
        orth_blocks = []
        for _ in range(num_qr):
            Z = torch.randn(in_features, in_features, generator=g)
            Q, R = torch.linalg.qr(Z)
            d = torch.diag(R).sign()
            d[d == 0] = 1.0
            orth_blocks.append(Q * d.unsqueeze(0))
        W_all = torch.cat(orth_blocks, dim=1)[:, :total_orth]
        self.register_buffer("Q_orth", W_all)
        self.register_buffer("knot_1", torch.linspace(0.15, 0.70, orth_dim).view(1, 1, orth_dim))
        self.register_buffer("knot_2", torch.linspace(0.70, 1.45, orth_dim).view(1, 1, orth_dim))

    def init_global_scale(self, f):
        if self.global_rms.item() == 0.0:
            rms_val = (f.float().norm(p=2, dim=-1) / math.sqrt(self.in_features)).mean().item()
            self.global_rms.fill_(max(rms_val, 1e-4))

    def forward(self, f):
        B = f.shape[0]
        self.init_global_scale(f)
        scale = self.global_rms.clamp(min=1.0)
        f_32 = f.float()
        f_scaled = f_32 / scale
        z = (f_scaled @ self.Q_orth.to(f.device)).view(B, self.num_heads, self.orth_dim)
        s0 = F.relu(z)
        s1 = F.relu(z - self.knot_1.to(f.device))
        s2 = F.relu(z - self.knot_2.to(f.device))
        psi_raw = (scale * self.kac_gain) * (s0 + self.alpha_spline * (0.45 * s1 + 0.25 * s2))
        psi_orth = psi_raw - psi_raw.mean(dim=-1, keepdim=True)
        f_exp = f_32.unsqueeze(1).expand(B, self.num_heads, self.in_features)
        return torch.cat([f_exp, psi_orth], dim=-1) # [B, 6, 2304]
'''

if "SOKAC_6Head_Projector" not in inc_net_code:
    # Thay forward của CosineLinear để hỗ trợ SO-KAC 6-Head
    target_fwd = '        if not self.use_RP:'
    replace_fwd = '''        if getattr(self, "use_sokac", False) and getattr(self, "sokac_weights", None) is not None:
            H = self.sokac_proj(input)
            W_curr = self.sokac_weights[:, :self.out_features, :].to(input.device)
            out = torch.einsum("bhd,hcd->bc", H, W_curr) / 6.0
            if self.sigma is not None:
                out = self.sigma * out
            return {'logits': out}
        if not self.use_RP:'''
    inc_net_code = inc_net_code.replace(target_fwd, replace_fwd)
    inc_net_code = sokac_class_code + "\n" + inc_net_code
    with open(inc_net_path, "w", encoding="utf-8") as f:
        f.write(inc_net_code)
    print("[OK] Đã tích hợp SO-KAC 6-Head Projector vào RanPAC/inc_net.py!")

# 3. Patch RanPAC.py: Tự động kích hoạt SO-KAC 6-Head Projector khi có flag use_sokac
ranpac_py_path = os.path.join(REPO_DIR, "RanPAC.py")
with open(ranpac_py_path, "r", encoding="utf-8") as f:
    ranpac_code = f.read()

if "from inc_net import SOKAC_6Head_Projector" not in ranpac_code:
    ranpac_code = "from inc_net import SOKAC_6Head_Projector\n" + ranpac_code
    
    # Patch setup_RP trong RanPAC.py
    target_setup = "        if self.args['M']>0:\n            #RP with M > 0\n            M=self.args['M']\n            self._network.fc.weight = nn.Parameter(torch.Tensor(self._network.fc.out_features, M).to(device='cuda')) #num classes in task x M\n            self._network.fc.reset_parameters()\n            self._network.fc.W_rand=torch.randn(self._network.fc.in_features,M).to(device='cuda')\n            self.W_rand=copy.deepcopy(self._network.fc.W_rand) #make a copy that gets passed each time the head is replaced"
    replace_setup = '''        if self.args.get('use_sokac', False):
            self._network.fc.use_sokac = True
            seed_val = self.args['seed'] if isinstance(self.args['seed'], int) else 1993
            self.sokac_proj = SOKAC_6Head_Projector(self._network.fc.in_features, seed=seed_val).to(device='cuda')
            self._network.fc.sokac_proj = self.sokac_proj
            self.G_sokac = torch.zeros(6, 2304, 2304)
            self.Q_sokac = torch.zeros(6, 2304, self.total_classnum)
            self.class_counts_sokac = torch.zeros(self.total_classnum, dtype=torch.float64)
            self.task_class_ranges = []
            self.sokac_weights = None
''' + target_setup
    ranpac_code = ranpac_code.replace(target_setup, replace_setup)
    
    # Patch replace_fc trong RanPAC.py
    target_replace_head = "            if self.args['M']>0:\n                self._network.fc.W_rand=self.W_rand"
    replace_replace_head = '''            if self.args.get('use_sokac', False):
                self._network.fc.use_sokac = True
                self._network.fc.sokac_proj = self.sokac_proj
                if getattr(self, 'sokac_weights', None) is not None:
                    self._network.fc.sokac_weights = self.sokac_weights
''' + target_replace_head
    ranpac_code = ranpac_code.replace(target_replace_head, replace_replace_head)
    
    # Patch feature projection trong replace_fc
    target_proj = "        if self.args['use_RP']:\n            #print('Number of pre-trained feature dimensions = ',Features_f.shape[-1])\n            if self.args['M']>0:\n                Features_h=torch.nn.functional.relu(Features_f@ self._network.fc.W_rand.cpu())\n            else:\n                Features_h=Features_f\n            self.Q=self.Q+Features_h.T @ Y \n            self.G=self.G+Features_h.T @ Features_h\n            ridge=self.optimise_ridge_parameter(Features_h,Y)\n            Wo=torch.linalg.solve(self.G+ridge*torch.eye(self.G.size(dim=0)),self.Q).T #better nmerical stability than .inv\n            self._network.fc.weight.data=Wo[0:self._network.fc.weight.shape[0],:].to(device='cuda')"
    replace_proj = '''        if self.args.get('use_sokac', False):
            self._network.fc.use_sokac = True
            self._network.fc.sokac_proj = self.sokac_proj
            
            # Ghi nhận dải lớp của task hiện tại cho Task Weight Aligning
            task_classes = torch.where(Y.sum(dim=0) > 0)[0]
            if len(task_classes) > 0:
                t_lo, t_hi = int(task_classes.min().item()), int(task_classes.max().item()) + 1
                if (t_lo, t_hi) not in self.task_class_ranges:
                    self.task_class_ranges.append((t_lo, t_hi))
            
            # Trích xuất đặc trưng SO-KAC 6-Head [N, 6, 2304]
            H = self.sokac_proj(Features_f.cuda()).cpu() # [N, 6, 2304]
            for h in range(6):
                H_h = H[:, h, :]
                self.G_sokac[h] += H_h.T @ H_h
                self.Q_sokac[h] += H_h.T @ Y
            self.class_counts_sokac += Y.to(torch.float64).sum(dim=0)
            
            # Class-Prior Balancing (gamma = 0.32) khi phát hiện mất cân bằng
            seen_mask = self.class_counts_sokac > 0
            cls_scale = torch.ones_like(self.class_counts_sokac)
            is_imbalanced = False
            if seen_mask.any():
                counts_seen = self.class_counts_sokac[seen_mask]
                cv_counts = (counts_seen.std() / (counts_seen.mean() + 1e-8)).item()
                if cv_counts > 0.08:
                    is_imbalanced = True
                    mean_count = counts_seen.mean()
                    cls_scale[seen_mask] = (mean_count / counts_seen).pow(0.32)
            
            ridge = self.optimise_ridge_parameter(H[:, 0, :], Y)
            
            # Giải Cholesky Float64 cho 6 heads
            I_mat = torch.eye(2304, dtype=torch.float64)
            W_list = []
            for h in range(6):
                G_reg = self.G_sokac[h].to(torch.float64) + ridge * I_mat
                Q_h = self.Q_sokac[h].to(torch.float64) * cls_scale.unsqueeze(0)
                L_h, info = torch.linalg.cholesky_ex(G_reg)
                if info.item() == 0:
                    Wh = torch.cholesky_solve(Q_h, L_h).T
                else:
                    Wh = torch.linalg.solve(G_reg, Q_h).T
                W_list.append(Wh.float())
            W_opt = torch.stack(W_list, dim=0) # [6, total_classnum, 2304]
            
            # Task Weight Aligning (beta = 0.80) chống thiên vị task mới
            eff_beta = 0.80 if is_imbalanced else 0.20
            if len(self.task_class_ranges) > 1 and eff_beta > 0.0:
                task_norms = []
                for (lo, hi) in self.task_class_ranges:
                    w_task = W_opt[:, lo:hi, :]
                    rms_task = w_task.pow(2).sum(dim=-1, keepdim=True).mean(dim=1, keepdim=True).sqrt()
                    task_norms.append(rms_task)
                mean_task_norm = torch.stack(task_norms, dim=0).mean(dim=0)
                for (lo, hi), r_t in zip(self.task_class_ranges, task_norms):
                    scale_t = (mean_task_norm / (r_t + 1e-8)).pow(eff_beta)
                    W_opt[:, lo:hi, :] *= scale_t
            
            self.sokac_weights = W_opt
            self._network.fc.sokac_weights = self.sokac_weights
        elif self.args['use_RP']:
            if self.args['M']>0:
                Features_h=torch.nn.functional.relu(Features_f@ self._network.fc.W_rand.cpu())
            else:
                Features_h=Features_f
            self.Q=self.Q+Features_h.T @ Y 
            self.G=self.G+Features_h.T @ Features_h
            ridge=self.optimise_ridge_parameter(Features_h,Y)
            Wo=torch.linalg.solve(self.G+ridge*torch.eye(self.G.size(dim=0)),self.Q).T
            self._network.fc.weight.data=Wo[0:self._network.fc.weight.shape[0],:].to(device='cuda')'''
    ranpac_code = ranpac_code.replace(target_proj, replace_proj)
    
    with open(ranpac_py_path, "w", encoding="utf-8") as f:
        f.write(ranpac_code)
    print("[OK] Đã tích hợp thành công SO-KAC 6-Head vào luồng giải RanPAC/RanPAC.py!")

# 4. Patch trainer.py: Lưu kết quả với tên run_tag tránh ghi đè
trainer_py_path = os.path.join(REPO_DIR, "trainer.py")
with open(trainer_py_path, "r", encoding="utf-8") as f:
    trainer_code = f.read()

target_save = "output_df.to_csv('./results/'+args['dataset']+'_publish_'+str(args['ID'])+'.csv')"
replace_save = "tag = args.get('run_tag', str(args['ID']))\n    output_df.to_csv('./results/'+args['dataset']+'_publish_'+str(tag)+'.csv')"
if target_save in trainer_code:
    trainer_code = trainer_code.replace(target_save, replace_save)
    with open(trainer_py_path, "w", encoding="utf-8") as f:
        f.write(trainer_code)
    print("[OK] Đã cập nhật RanPAC/trainer.py lưu file theo run_tag!")

In [ ]:
# ==============================================================================
# CELL 5: CHẠY BENCHMARK TOÀN DIỆN TRÊN STANFORD CARS-196 (3 SEEDS [0, 1, 2])
# ==============================================================================
print("\n" + "="*85)
print("🚀 [1/4] CHẠY CARS-196: ĐỐI CHỨNG RANPAC 10K VS SO-KAC TRÊN 2 BỘ WEIGHTS")
print("="*85)

# 1. ViT-B/16 IN21K: RanPAC 10k
print("\n[*] [CARS196 - IN21K] Chạy RanPAC 10k Official (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cars -i 4 --backbone pretrained_vit_b16_224_in21k --seeds 0 1 2

# 2. ViT-B/16 IN21K: SO-KAC Ridge 6-Head (Main Proposal)
print("\n[*] [CARS196 - IN21K] Chạy SO-KAC Ridge 6-Head (6x2304d, 3-ReLU) (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cars -i 4 --backbone pretrained_vit_b16_224_in21k --sokac --seeds 0 1 2

# 3. ViT-B/16 IN1K: RanPAC 10k
print("\n[*] [CARS196 - IN1K] Chạy RanPAC 10k Official (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cars -i 4 --backbone pretrained_vit_b16_224 --seeds 0 1 2

# 4. ViT-B/16 IN1K: SO-KAC Ridge 6-Head (Main Proposal)
print("\n[*] [CARS196 - IN1K] Chạy SO-KAC Ridge 6-Head (6x2304d, 3-ReLU) (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cars -i 4 --backbone pretrained_vit_b16_224 --sokac --seeds 0 1 2

In [ ]:
# ==============================================================================
# CELL 6: CHẠY BENCHMARK TOÀN DIỆN TRÊN CUB-200 (3 SEEDS [0, 1, 2])
# ==============================================================================
print("\n" + "="*85)
print("🚀 [2/4] CHẠY CUB-200: ĐỐI CHỨNG RANPAC 10K VS SO-KAC TRÊN 2 BỘ WEIGHTS")
print("="*85)

# 1. ViT-B/16 IN21K: RanPAC 10k
print("\n[*] [CUB200 - IN21K] Chạy RanPAC 10k Official (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cub -i 4 --backbone pretrained_vit_b16_224_in21k --seeds 0 1 2

# 2. ViT-B/16 IN21K: SO-KAC Ridge 6-Head (Main Proposal)
print("\n[*] [CUB200 - IN21K] Chạy SO-KAC Ridge 6-Head (6x2304d, 3-ReLU) (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cub -i 4 --backbone pretrained_vit_b16_224_in21k --sokac --seeds 0 1 2

# 3. ViT-B/16 IN1K: RanPAC 10k
print("\n[*] [CUB200 - IN1K] Chạy RanPAC 10k Official (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cub -i 4 --backbone pretrained_vit_b16_224 --seeds 0 1 2

# 4. ViT-B/16 IN1K: SO-KAC Ridge 6-Head (Main Proposal)
print("\n[*] [CUB200 - IN1K] Chạy SO-KAC Ridge 6-Head (6x2304d, 3-ReLU) (Seeds 0, 1, 2)...")
!cd RanPAC && python main.py -d cub -i 4 --backbone pretrained_vit_b16_224 --sokac --seeds 0 1 2

In [ ]:
# ==============================================================================
# CELL 7: CHẠY BENCHMARK TRÊN CIFAR-100 & IMAGENET-R (3 SEEDS [0, 1, 2])
# ==============================================================================
print("\n" + "="*85)
print("🚀 [3/4] CHẠY CIFAR-100 & [4/4] IMAGENET-R")
print("="*85)

# CIFAR-100
print("\n[*] [CIFAR100 - IN21K] RanPAC 10k...")
!cd RanPAC && python main.py -d cifar224 -i 4 --backbone pretrained_vit_b16_224_in21k --seeds 0 1 2
print("\n[*] [CIFAR100 - IN21K] SO-KAC Ridge 6-Head...")
!cd RanPAC && python main.py -d cifar224 -i 4 --backbone pretrained_vit_b16_224_in21k --sokac --seeds 0 1 2

# ImageNet-R
print("\n[*] [ImageNet-R - IN1K] RanPAC 10k...")
!cd RanPAC && python main.py -d imagenetr -i 4 --backbone pretrained_vit_b16_224 --seeds 0 1 2
print("\n[*] [ImageNet-R - IN1K] SO-KAC Ridge 6-Head...")
!cd RanPAC && python main.py -d imagenetr -i 4 --backbone pretrained_vit_b16_224 --sokac --seeds 0 1 2

In [ ]:
# ==============================================================================
# CELL 8: BẢNG TỔNG KẾT MASTER: ĐỐI CHỨNG CÔNG KHAI 2 METRIC LAST & AVG
# ==============================================================================
import glob
import pandas as pd
import numpy as np

results_dir = os.path.join(REPO_DIR, "results")
csv_files = glob.glob(os.path.join(results_dir, "*.csv"))
csv_files = [f for f in csv_files if "class_preds" not in f]

print("\n" + "="*95)
print("🏆 BẢNG KẾT QUẢ ĐỐI CHỨNG MASTER: REPLICATE 100% REPO GỐC RANPAC VS SO-KAC")
print("="*95)
print(f"{'Tập Tin Kết Quả':45s} | {'Last A_T (%)':18s} | {'Avg A_avg (%)':18s}")
print("-"*95)

summary_rows = []
for fpath in sorted(csv_files):
    fname = os.path.basename(fpath)
    try:
        df = pd.read_csv(fpath)
        if "top1_total" in df.columns and "ave_acc" in df.columns:
            last_val = df["top1_total"].iloc[-1]
            avg_val  = df["ave_acc"].iloc[-1]
            print(f"{fname:45s} | {last_val:6.2f}%            | {avg_val:6.2f}%")
            summary_rows.append({"file": fname, "Last": last_val, "Avg": avg_val})
    except Exception as e:
        print(f"{fname:45s} | Lỗi đọc file: {e}")

print("="*95)

In [ ]:
# ==============================================================================
# CELL 9: (1-CLICK) NÉN TOÀN BỘ KẾT QUẢ & LOGS ĐỂ TẢI VỀ MÁY
# ==============================================================================
import zipfile

zip_out = "/kaggle/working/RanPAC_Official_Benchmark_Results.zip" if IS_KAGGLE else "RanPAC_Official_Benchmark_Results.zip"
res_folder = os.path.join(REPO_DIR, "results")
logs_folder = os.path.join(REPO_DIR, "logs")

with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in [res_folder, logs_folder]:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    full_p = os.path.join(root, file)
                    rel_p = os.path.relpath(full_p, REPO_DIR)
                    zipf.write(full_p, rel_p)

print(f"[OK] Đã nén toàn bộ kết quả CSV và Logs vào file: {zip_out}")
if IS_KAGGLE:
    print("[*] Bạn có thể tải file zip này về máy chỉ với 1 cú click trong thư mục Output của Kaggle!")